# Day 13 — Gold Dims (corrected)

Builds all dimension tables from **corrected** Silver columns (matched to actual Bronze JSON keys).

| Dim | Source | SCD | Key columns used |
|---|---|---|---|
| DimState | api/states | 1 | state_code, name, country |
| DimCity | api/cities | 1 | city_id, city_name, state_code, postcode |
| DimStation | api/stations | 1 | station_id, name, state_code, city, lat/lng, charger_type, max_power_kw, num_connectors, operator, is_active |
| DimCharger | api/chargers | 2 | charger_id, station_id, connector_type, max_kw, firmware_version, status — valid_from/valid_to from updated_at |
| DimCustomer | api/customers | 2 | customer_id, full_name, email, phone, loyalty_tier, signup_date |
| DimVehicle | api/vehicles | 1 | vehicle_id, make, model, year, battery_capacity_kwh, range_km, vehicle_type, registration_state, partner_id |
| DimEmployee | api/employees | 1 | employee_id, name, department, station_id, state, hire_date |
| DimPartner | api/partners | 1 | partner_id, partner_name, state, status, revenue_share_pct, contract_start, contract_end |
| DimChargeCard | api/charge_cards | 1 | card_id, customer_id, rfid_uid, card_number_masked, card_type, status |
| DimTariff | api/tariffs | 2 | tariff_id, version, peak_offpeak, rate_per_kwh, effective_from, effective_to, is_current |
| DimTime | generated | — | 2020-2030 hour grain |


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
print('Imports OK')

In [ ]:
SILVER  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD    = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dims'
PIPELINE = 'pl_gold_dims_v1'
RUN_TS   = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def silver(name):
    return spark.read.format('delta').load(f'{SILVER}/{name}')

def write_dim(df, name):
    path = f'{GOLD}/{name}'
    (df.write.format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .save(path))
    print(f'  {name:<25} {df.count():>7} rows -> {path}')

print(f'GOLD : {GOLD}\nRUN  : {RUN_TS}')

In [ ]:
# DimState — Type 1
# Silver cols: state_code, name, country, created_at, updated_at
dim_state = (
    silver('states')
    .select(
        F.col('state_code'),
        F.trim(F.col('name')).alias('state_name'),
        F.trim(F.col('country')).alias('country'),
    )
    .filter(F.col('state_code').isNotNull() & (F.col('state_code') != ''))
    .distinct()
    .withColumn('state_key',        F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('state_key', 'state_code', 'state_name', 'country',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_state, 'DimState')

In [ ]:
# DimCity — Type 1
# Silver cols: city_id, city_name, state_code, postcode, created_at, updated_at
dim_city = (
    silver('cities')
    .select(
        F.col('city_id'),
        F.trim(F.col('city_name')).alias('city_name'),
        F.trim(F.col('state_code')).alias('state_code'),
        F.col('postcode'),
    )
    .filter(F.col('city_id').isNotNull())
    .withColumn('city_key',         F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('city_key', 'city_id', 'city_name', 'state_code', 'postcode',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_city, 'DimCity')

In [ ]:
# DimStation — Type 1
# Silver cols: station_id, name, state_code, city, latitude, longitude,
#              charger_type, max_power_kw, num_connectors, operator,
#              is_active, commissioned_date, created_at, updated_at
dim_station = (
    silver('stations')
    .select(
        F.col('station_id'),
        F.trim(F.col('name')).alias('station_name'),
        F.trim(F.col('state_code')).alias('state_code'),
        F.trim(F.col('city')).alias('city'),
        F.col('latitude'),
        F.col('longitude'),
        F.trim(F.col('charger_type')).alias('charger_type'),
        F.col('max_power_kw'),
        F.col('num_connectors'),
        F.trim(F.col('operator')).alias('operator'),
        F.col('is_active'),
        F.col('commissioned_date'),
    )
    .filter(F.col('station_id').isNotNull())
    .withColumn('station_key',      F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_id', 'station_name', 'state_code', 'city',
            'latitude', 'longitude', 'charger_type', 'max_power_kw',
            'num_connectors', 'operator', 'is_active', 'commissioned_date',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_station, 'DimStation')

In [ ]:
# DimCharger — SCD Type 2
# Silver cols: charger_id, station_id, connector_type, max_kw,
#              firmware_version, install_date, status, created_at, updated_at
# SCD2: valid_from = created_at, valid_to = NULL (current), is_current = True
# On re-runs with MERGE this would add expired rows; here we do full overwrite (initial load).
dim_charger = (
    silver('chargers')
    .select(
        F.col('charger_id'),
        F.col('station_id'),
        F.trim(F.col('connector_type')).alias('connector_type'),
        F.col('max_kw'),
        F.col('firmware_version'),
        F.col('install_date'),
        F.trim(F.col('status')).alias('status'),
        F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'),
        F.lit(True).alias('is_current'),
    )
    .filter(F.col('charger_id').isNotNull())
    .withColumn('charger_key',      F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('charger_key', 'charger_id', 'station_id', 'connector_type',
            'max_kw', 'firmware_version', 'install_date', 'status',
            'valid_from', 'valid_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_charger, 'DimCharger')

In [ ]:
# DimCustomer — SCD Type 2
# Silver cols: customer_id, full_name, email, phone,
#              loyalty_tier, signup_date, created_at, updated_at
# SCD2: valid_from = created_at, valid_to = NULL, is_current = True
dim_customer = (
    silver('customers')
    .select(
        F.col('customer_id'),
        F.trim(F.col('full_name')).alias('full_name'),
        F.trim(F.col('email')).alias('email'),
        F.trim(F.col('phone')).alias('phone'),
        F.trim(F.col('loyalty_tier')).alias('loyalty_tier'),
        F.col('signup_date'),
        F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'),
        F.lit(True).alias('is_current'),
    )
    .filter(F.col('customer_id').isNotNull())
    .withColumn('customer_key',     F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('customer_key', 'customer_id', 'full_name', 'email', 'phone',
            'loyalty_tier', 'signup_date',
            'valid_from', 'valid_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_customer, 'DimCustomer')

In [ ]:
# DimVehicle — Type 1
# Silver cols: vehicle_id, make, model, year, battery_capacity_kwh,
#              range_km, vehicle_type, registration_state, partner_id,
#              effective_from, effective_to, is_current, created_at, updated_at
dim_vehicle = (
    silver('vehicles')
    .select(
        F.col('vehicle_id'),
        F.trim(F.col('make')).alias('make'),
        F.trim(F.col('model')).alias('model'),
        F.col('year'),
        F.col('battery_capacity_kwh'),
        F.col('range_km'),
        F.trim(F.col('vehicle_type')).alias('vehicle_type'),
        F.trim(F.col('registration_state')).alias('registration_state'),
        F.col('partner_id'),
    )
    .filter(F.col('vehicle_id').isNotNull())
    .withColumn('vehicle_key',      F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('vehicle_key', 'vehicle_id', 'make', 'model', 'year',
            'battery_capacity_kwh', 'range_km', 'vehicle_type',
            'registration_state', 'partner_id',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_vehicle, 'DimVehicle')

In [ ]:
# DimEmployee — Type 1
# Silver cols: employee_id, name, department, station_id, state, hire_date,
#              created_at, updated_at
dim_employee = (
    silver('employees')
    .select(
        F.col('employee_id'),
        F.trim(F.col('name')).alias('full_name'),
        F.trim(F.col('department')).alias('department'),
        F.col('station_id'),
        F.trim(F.col('state')).alias('state'),
        F.col('hire_date'),
    )
    .filter(F.col('employee_id').isNotNull())
    .withColumn('employee_key',     F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('employee_key', 'employee_id', 'full_name', 'department',
            'station_id', 'state', 'hire_date',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_employee, 'DimEmployee')

In [ ]:
# DimPartner — Type 1
# Silver cols: partner_id, partner_name, state, status,
#              revenue_share_pct, contract_start, contract_end,
#              created_at, updated_at
dim_partner = (
    silver('partners')
    .select(
        F.col('partner_id'),
        F.trim(F.col('partner_name')).alias('partner_name'),
        F.trim(F.col('state')).alias('state'),
        F.trim(F.col('status')).alias('status'),
        F.col('revenue_share_pct'),
        F.col('contract_start'),
        F.col('contract_end'),
    )
    .filter(F.col('partner_id').isNotNull())
    .withColumn('partner_key',      F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('partner_key', 'partner_id', 'partner_name', 'state', 'status',
            'revenue_share_pct', 'contract_start', 'contract_end',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_partner, 'DimPartner')

In [ ]:
# DimChargeCard — Type 1
# Silver cols: card_id, customer_id, rfid_uid, card_number,
#              card_type, status, created_at, updated_at
# card_number is masked in Gold: first 4 + ****-**** + last 4
dim_card = (
    silver('charge_cards')
    .select(
        F.col('card_id'),
        F.col('customer_id'),
        F.col('rfid_uid'),
        F.when(
            F.col('card_number').isNotNull() & (F.length('card_number') >= 8),
            F.concat(
                F.substring('card_number', 1, 4),
                F.lit('-****-****-'),
                F.substring('card_number', -4, 4)
            )
        ).otherwise(F.lit('****')).alias('card_number_masked'),
        F.trim(F.col('card_type')).alias('card_type'),
        F.trim(F.col('status')).alias('status'),
    )
    .filter(F.col('card_id').isNotNull())
    .withColumn('card_key',         F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('card_key', 'card_id', 'customer_id', 'rfid_uid',
            'card_number_masked', 'card_type', 'status',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_card, 'DimChargeCard')

In [ ]:
# DimTariff — SCD Type 2
# Silver cols: tariff_id, version, peak_offpeak, rate_per_kwh,
#              effective_from, effective_to, is_current, created_at, updated_at
# is_current comes directly from source (no need to compute)
dim_tariff = (
    silver('tariffs')
    .select(
        F.col('tariff_id'),
        F.col('version'),
        F.trim(F.col('peak_offpeak')).alias('peak_offpeak'),
        F.col('rate_per_kwh'),
        F.col('effective_from'),
        F.col('effective_to'),
        F.col('is_current'),
    )
    .filter(F.col('tariff_id').isNotNull())
    .withColumn('tariff_key',       F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('tariff_key', 'tariff_id', 'version', 'peak_offpeak', 'rate_per_kwh',
            'effective_from', 'effective_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_tariff, 'DimTariff')

In [ ]:
# DimTime — generated, hour grain, 2020-2030
# No source. Fully computed.
# time_key format: YYYYMMDDHHH (e.g. 2026071914)
# financial_year_au: AU FY starts 1 Jul (Jul 2025–Jun 2026 = FY2026)

TOTAL_HOURS = 365 * 24 * 11  # 2020-2030 inclusive (approx, leap years ignored)

dim_time = (
    spark.range(TOTAL_HOURS)
    .select(
        F.expr(
            "timestamp '2020-01-01 00:00:00' "
            "+ make_interval(0, 0, 0, 0, cast(id as int), 0, 0)"
        ).alias('ts')
    )
    .select(
        F.concat_ws('',
            F.year('ts').cast('string'),
            F.lpad(F.month('ts').cast('string'),      2, '0'),
            F.lpad(F.dayofmonth('ts').cast('string'), 2, '0'),
            F.lpad(F.hour('ts').cast('string'),       2, '0'),
        ).alias('time_key'),
        F.col('ts'),
        F.to_date('ts').alias('date'),
        F.year('ts').alias('year'),
        F.month('ts').alias('month'),
        F.dayofmonth('ts').alias('day'),
        F.hour('ts').alias('hour'),
        F.date_format('ts', 'EEEE').alias('day_of_week'),
        F.when(F.dayofweek('ts').isin(1, 7), True).otherwise(False).alias('is_weekend'),
        F.quarter('ts').alias('quarter'),
        F.when(
            F.month('ts') >= 7,
            F.concat(F.lit('FY'), (F.year('ts') + 1).cast('string'))
        ).otherwise(
            F.concat(F.lit('FY'), F.year('ts').cast('string'))
        ).alias('financial_year_au'),
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

write_dim(dim_time, 'DimTime')

In [ ]:
# Summary
DIMS = [
    'DimState', 'DimCity', 'DimStation', 'DimCharger',
    'DimCustomer', 'DimVehicle', 'DimEmployee', 'DimPartner',
    'DimChargeCard', 'DimTariff', 'DimTime',
]
print('=' * 55)
print('GOLD DIMS SUMMARY')
print('=' * 55)
for d in DIMS:
    path = f'{GOLD}/{d}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        print(f'  {d:<25} {cnt:>8} rows')
    except Exception as e:
        print(f'  {d:<25} ERROR: {e}')
print(f'\nRUN_TS : {RUN_TS}')